# Fase 2b: Cascade Predictivo — Pasos 2, 3 y 4

**Scope:** Entrenar los 3 modelos restantes del cascade predictivo:

```
PASO 1: ¿Canjea? → XGBoost multiclase → P(y=0), P(y=1), P(y=2)     [fase2a]
    │
    ├── Si P(y=1)+P(y=2) > 0.3 (threshold ajustable)
    │
PASO 2: ¿Dónde? → XGBoost multiclase → P(retailer)                  [este notebook]
    │
PASO 3: ¿Cuánto? → XGBoost regresión → Monto estimado (puntos)      [este notebook]
    │
PASO 4: ¿Revenue? → XGBoost regresión → Revenue estimado ($CLP)     [este notebook]
```

**Prerequisito:** `fase2_eda_modelo.ipynb` (Paso 1) completado.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    f1_score, accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, r2_score, mean_absolute_percentage_error,
    mean_squared_error, roc_auc_score
)
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import shap

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

print("Imports OK")

In [ ]:
# ── Carga de datos ─────────────────────────────────────────────────────────
USE_MOCK = True

if USE_MOCK:
    import duckdb
    with open("../../fase1/test_mock_local.py") as f:
        code = f.read().replace("con.close()", "# con.close()")
    _globals = {}
    exec(code, _globals)
    con = _globals['con']
    df = con.execute("SELECT * FROM customer_snapshot").df()
    print(f"Datos cargados desde mock DuckDB: {df.shape}")
else:
    from google.cloud import bigquery
    client = bigquery.Client(project="my-gcp-project")
    df = client.query("SELECT * FROM `my-gcp-project.loyalty_analytics.customer_snapshot`").to_dataframe()
    con = None
    print(f"Datos cargados desde BigQuery: {df.shape}")

df['t0'] = pd.to_datetime(df['t0'])
print(f"Shape: {df.shape}, t0s: {sorted(df['t0'].dt.strftime('%Y-%m').unique())}")

In [ ]:
# ── Reusar clasificacion de columnas de fase2a ───────────────────────
ID_COLS = ['cust_id', 't0', 'fecha_proceso']
TARGET_COL_PASO1 = 'y'
TARGET_RELATED = ['canjea_post', 'n_canjes_post', 'revenue_post_12m', 'has_redeemed_before_t0',
                  'retailer_post', 'monto_redeem_post']  # defensive: exclude even though added later

CATEGORICAL_FEATURES = ['tier', 'gender', 'city', 'dominant_retailer',
                        'funnel_state_at_t0', 'status']
BOOLEAN_FEATURES = [
    'cust_active_card_flg', 'cust_active_deb_flg', 'cust_active_omp_flg',
    'contact_email_flg', 'contact_phone_flg', 'contact_push_flg',
    'redeem_capacity', 'is_cyber_month', 'is_holiday_month'
]

EXCLUDED = set(ID_COLS + [TARGET_COL_PASO1] + TARGET_RELATED + CATEGORICAL_FEATURES + BOOLEAN_FEATURES)
NUMERIC_FEATURES = [c for c in df.columns if c not in EXCLUDED]

# Feature engineering — retailer_entropy frequency-based (matching production SQL)
freqs = df[['freq_store_a', 'freq_store_b', 'freq_store_c', 'freq_store_d', 'freq_store_e']].values
tot = np.where(freqs.sum(axis=1, keepdims=True) == 0, 1, freqs.sum(axis=1, keepdims=True))
p = freqs / tot
df['retailer_entropy'] = -(p * np.where(p > 0, np.log(p), 0)).sum(axis=1)

# NOTE: engagement_score removed — it was a weighted combination of frequency_total,
# recency_days, and retailer_entropy that introduced min_max normalization leakage
# (computed on full data before split). XGBoost can learn this combination natively.

# earn_acceleration: ratio velocidad reciente vs largo plazo
# > 1 = acelerando, < 1 = desacelerando
df['earn_acceleration'] = np.where(
    df['earn_velocity_90'] > 0,
    df['earn_velocity_30'] / df['earn_velocity_90'],
    0
)

NUMERIC_FEATURES += ['retailer_entropy', 'earn_acceleration']
# burstiness se agrega despues (requiere query a DuckDB)

print(f"Features iniciales: {len(NUMERIC_FEATURES)} numericas + {len(CATEGORICAL_FEATURES)} cat + {len(BOOLEAN_FEATURES)} bool")

## 1. Computar Targets para Pasos 2, 3 y 4

- **Paso 2 target** (`retailer_post`): retailer dominante en canjes post-t0
- **Paso 3 target** (`monto_redeem_post`): total puntos canjeados post-t0
- **Paso 4 target** (`revenue_post_12m`): ya existe en snapshot

In [ ]:
# ── Computar targets Paso 2 y 3 + burstiness desde tablas raw ────────
if USE_MOCK:
    # Paso 2: retailer dominante post-t0 (el que mas canjes tiene)
    target_retailer = con.execute("""
        WITH base_t0s AS (
            SELECT DISTINCT cust_id, t0, t0 + INTERVAL 12 MONTH AS post_end
            FROM customer_snapshot
        ),
        redeems_post AS (
            SELECT b.cust_id, b.t0, r.channel_name,
                   COUNT(*) AS n_canjes,
                   SUM(r.redemption_points_amt) AS monto_pts
            FROM base_t0s b
            JOIN mock_redemption_entity r
              ON  r.cust_id = b.cust_id
              AND r.redemption_date >= b.t0
              AND r.redemption_date < b.post_end
              AND r.return_flag = FALSE
            GROUP BY b.cust_id, b.t0, r.channel_name
        ),
        ranked AS (
            SELECT *, ROW_NUMBER() OVER (PARTITION BY cust_id, t0 ORDER BY n_canjes DESC, monto_pts DESC) AS rn
            FROM redeems_post
        )
        SELECT cust_id, t0, channel_name AS retailer_post
        FROM ranked WHERE rn = 1
    """).df()

    # Paso 3: monto total puntos canjeados post-t0
    target_monto = con.execute("""
        WITH base_t0s AS (
            SELECT DISTINCT cust_id, t0, t0 + INTERVAL 12 MONTH AS post_end
            FROM customer_snapshot
        )
        SELECT b.cust_id, b.t0,
               SUM(r.redemption_points_amt) AS monto_redeem_post
        FROM base_t0s b
        JOIN mock_redemption_entity r
          ON  r.cust_id = b.cust_id
          AND r.redemption_date >= b.t0
          AND r.redemption_date < b.post_end
          AND r.return_flag = FALSE
        GROUP BY b.cust_id, b.t0
    """).df()

    # Burstiness: std(dias entre compras) / mean(dias entre compras) por cliente×t0
    burstiness_df = con.execute("""
        WITH base_t0s AS (
            SELECT DISTINCT cust_id, t0
            FROM customer_snapshot
        ),
        txns_pre AS (
            SELECT b.cust_id, b.t0, t.tran_date
            FROM base_t0s b
            JOIN mock_transaction_entity t
              ON  t.cust_id = b.cust_id
              AND t.tran_date < b.t0
              AND t.tran_valid_flg = 1
            ORDER BY b.cust_id, b.t0, t.tran_date
        ),
        with_lag AS (
            SELECT *, tran_date - LAG(tran_date) OVER (PARTITION BY cust_id, t0 ORDER BY tran_date) AS days_gap
            FROM txns_pre
        ),
        agg AS (
            SELECT cust_id, t0,
                   AVG(CAST(days_gap AS FLOAT)) AS mean_gap,
                   STDDEV_POP(CAST(days_gap AS FLOAT)) AS std_gap
            FROM with_lag
            WHERE days_gap IS NOT NULL
            GROUP BY cust_id, t0
        )
        SELECT cust_id, t0,
               CASE WHEN mean_gap > 0 THEN std_gap / mean_gap ELSE 0 END AS burstiness
        FROM agg
    """).df()

    con.close()
    print("Conexion DuckDB cerrada")

else:
    raise NotImplementedError("Agregar queries BigQuery para targets y burstiness")

# Merge targets al df
target_retailer['t0'] = pd.to_datetime(target_retailer['t0'])
target_monto['t0'] = pd.to_datetime(target_monto['t0'])
burstiness_df['t0'] = pd.to_datetime(burstiness_df['t0'])

df = df.merge(target_retailer, on=['cust_id', 't0'], how='left')
df = df.merge(target_monto, on=['cust_id', 't0'], how='left')
df = df.merge(burstiness_df, on=['cust_id', 't0'], how='left')
df['burstiness'] = df['burstiness'].fillna(0)

# Agregar burstiness a features
NUMERIC_FEATURES.append('burstiness')
FEATURE_COLS = CATEGORICAL_FEATURES + BOOLEAN_FEATURES + NUMERIC_FEATURES

print(f"Total features: {len(FEATURE_COLS)}")
print(f"retailer_post: {df['retailer_post'].notna().sum()} no-null de {len(df)}")
print(f"monto_redeem_post: {df['monto_redeem_post'].notna().sum()} no-null")
print(f"burstiness: mean={df['burstiness'].mean():.3f}, median={df['burstiness'].median():.3f}")
print(f"earn_acceleration: mean={df['earn_acceleration'].mean():.3f}, median={df['earn_acceleration'].median():.3f}")

In [ ]:
# ── EDA rapido de targets cascade ─────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Paso 2: distribucion retailer_post
df_canje = df[df['canjea_post'] == True].copy()
vc_ret = df_canje['retailer_post'].value_counts()
vc_ret.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title(f'Paso 2: retailer_post (n={len(df_canje)})')
axes[0].set_ylabel('N')

# Paso 3: distribucion monto
df_canje['monto_redeem_post'].hist(bins=30, ax=axes[1], color='coral')
axes[1].set_title(f'Paso 3: monto_redeem_post (puntos)')
axes[1].axvline(df_canje['monto_redeem_post'].median(), color='black', ls='--', label=f"median={df_canje['monto_redeem_post'].median():,.0f}")
axes[1].legend()

# Paso 4: distribucion revenue
df['revenue_post_12m'].hist(bins=30, ax=axes[2], color='seagreen')
axes[2].set_title(f'Paso 4: revenue_post_12m ($CLP)')
axes[2].axvline(df['revenue_post_12m'].median(), color='black', ls='--', label=f"median={df['revenue_post_12m'].median():,.0f}")
axes[2].legend()

plt.tight_layout()
plt.show()

print(f"\nResumen targets:")
print(f"  Paso 2 — retailer_post: {vc_ret.to_dict()}")
print(f"  Paso 3 — monto: mean={df_canje['monto_redeem_post'].mean():,.0f}, median={df_canje['monto_redeem_post'].median():,.0f}, std={df_canje['monto_redeem_post'].std():,.0f}")
print(f"  Paso 4 — revenue: mean={df['revenue_post_12m'].mean():,.0f}, median={df['revenue_post_12m'].median():,.0f}, zeros={( df['revenue_post_12m']==0).sum()}")

In [ ]:
# ── Split temporal (mismo que fase2a) ───────────────────────────────
if USE_MOCK:
    TRAIN_END = pd.Timestamp('2024-07-01')
    VAL_END   = pd.Timestamp('2025-01-01')
else:
    TRAIN_END = pd.Timestamp('2024-10-01')
    VAL_END   = pd.Timestamp('2025-01-01')

df_train = df[df['t0'] < TRAIN_END].copy()
df_val   = df[(df['t0'] >= TRAIN_END) & (df['t0'] < VAL_END)].copy()
df_test  = df[df['t0'] >= VAL_END].copy()

assert len(set(df_train['t0']) & set(df_test['t0'])) == 0
assert len(set(df_val['t0']) & set(df_test['t0'])) == 0
assert len(df_val) > 0 and len(df_test) > 0

for name, subset in [('TRAIN', df_train), ('VAL', df_val), ('TEST', df_test)]:
    t0s = sorted(subset['t0'].dt.strftime('%Y-%m').unique())
    n_canje = (subset['canjea_post'] == True).sum()
    print(f"{name}: {len(subset):,} filas | canjean: {n_canje} ({n_canje/len(subset)*100:.1f}%) | t0s={t0s}")

In [ ]:
# ── Encoding de features (fit solo en TRAIN, reusar en val/test) ────
ord_enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
ord_enc.fit(df_train[CATEGORICAL_FEATURES])

for subset in [df_train, df_val, df_test]:
    subset[CATEGORICAL_FEATURES] = ord_enc.transform(subset[CATEGORICAL_FEATURES])
    for col in BOOLEAN_FEATURES:
        subset[col] = subset[col].astype(int)

print(f"Encoding completado. Features: {len(FEATURE_COLS)}")

---
## 2. Paso 2: ¿Dónde canjea? (Retailer)

XGBoost multiclase sobre **solo canjeadores** (canjea_post=True).  
Target: retailer dominante del canje post-t0 (4 clases: STOREA, STOREB, STOREC, STORED).  
En inferencia se aplica solo a clientes con P(canje) > 0.3.

| Métrica | Mínimo | Objetivo |
|---------|--------|----------|
| Accuracy | > 0.60 | > 0.75 |
| Top-2 accuracy | > 0.85 | > 0.93 |

In [ ]:
# ── Paso 2: Preparar datos (solo canjeadores) ────────────────────
# Filtrar a canjeadores y excluir STOREE (solo 4 retailers en el modelo)
RETAILERS_TARGET = ['STOREA', 'STOREB', 'STOREC', 'STORED']

tr2 = df_train[df_train['retailer_post'].isin(RETAILERS_TARGET)].copy()
va2 = df_val[df_val['retailer_post'].isin(RETAILERS_TARGET)].copy()
te2 = df_test[df_test['retailer_post'].isin(RETAILERS_TARGET)].copy()

# Encode target
le_ret = LabelEncoder()
le_ret.fit(RETAILERS_TARGET)

X_tr2 = tr2[FEATURE_COLS].values.astype(np.float32)
y_tr2 = le_ret.transform(tr2['retailer_post'])
X_va2 = va2[FEATURE_COLS].values.astype(np.float32)
y_va2 = le_ret.transform(va2['retailer_post'])
X_te2 = te2[FEATURE_COLS].values.astype(np.float32)
y_te2 = le_ret.transform(te2['retailer_post'])

# Sample weights
vc2 = pd.Series(y_tr2).value_counts()
w_map2 = {c: len(y_tr2) / (len(vc2) * n) for c, n in vc2.items()}
w_tr2 = np.array([w_map2[y] for y in y_tr2])

print(f"Paso 2 — Train: {len(tr2)}, Val: {len(va2)}, Test: {len(te2)}")
print(f"Clases: {dict(zip(le_ret.classes_, [int(x) for x in le_ret.transform(le_ret.classes_)]))}")
print(f"Distribucion train: {pd.Series(y_tr2).value_counts().sort_index().to_dict()}")

In [ ]:
# ── Paso 2: Optuna ────────────────────────────────────────────────
def objective_paso2(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 800),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10, log=True),
        'tree_method': 'hist',
        'objective': 'multi:softprob',
        'num_class': len(RETAILERS_TARGET),
        'eval_metric': 'mlogloss',
        'random_state': 42,
        'verbosity': 0,
        'early_stopping_rounds': 50,
    }
    model = xgb.XGBClassifier(**params)
    model.fit(X_tr2, y_tr2, sample_weight=w_tr2,
              eval_set=[(X_va2, y_va2)], verbose=False)
    y_pred = model.predict(X_va2)
    return f1_score(y_va2, y_pred, average='macro')

study2 = optuna.create_study(direction='maximize', study_name='paso2_retailer')
study2.optimize(objective_paso2, n_trials=30, show_progress_bar=True)

print(f"\nBest F1-macro (val): {study2.best_value:.4f}")
print(f"Best params: {study2.best_params}")

In [ ]:
# ── Paso 2: Modelo final + evaluacion ────────────────────────────
best_p2 = study2.best_params.copy()
best_p2.update({'tree_method': 'hist', 'objective': 'multi:softprob',
                'num_class': len(RETAILERS_TARGET), 'eval_metric': 'mlogloss',
                'random_state': 42, 'verbosity': 0, 'early_stopping_rounds': 50})

model2 = xgb.XGBClassifier(**best_p2)
model2.fit(X_tr2, y_tr2, sample_weight=w_tr2,
           eval_set=[(X_va2, y_va2)], verbose=False)

# Predictions
y_pred2_test = model2.predict(X_te2)
y_proba2_test = model2.predict_proba(X_te2)

# Accuracy
acc2 = accuracy_score(y_te2, y_pred2_test)

# Top-2 accuracy
top2_preds = np.argsort(y_proba2_test, axis=1)[:, -2:]
top2_acc = np.mean([y_te2[i] in top2_preds[i] for i in range(len(y_te2))])

print(f"TEST — Accuracy: {acc2:.4f}, Top-2 Accuracy: {top2_acc:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_te2, y_pred2_test, target_names=le_ret.classes_))

# Confusion matrix
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(y_te2, y_pred2_test,
    display_labels=le_ret.classes_, normalize='true', ax=ax, cmap='Blues')
ax.set_title(f'Paso 2: Confusion Matrix (TEST) \u2014 Acc={acc2:.3f}, Top2={top2_acc:.3f}')
plt.tight_layout()
plt.show()

# Metricas vs umbrales
print(f"\n{'Metrica':<25} {'Minimo':<10} {'Objetivo':<10} {'Resultado':<10} {'Status'}")
print('=' * 65)
print(f"{'Accuracy':<25} {'>0.60':<10} {'>0.75':<10} {acc2:<10.4f} {'OK' if acc2 > 0.60 else 'FAIL'}")
print(f"{'Top-2 Accuracy':<25} {'>0.85':<10} {'>0.93':<10} {top2_acc:<10.4f} {'OK' if top2_acc > 0.85 else 'FAIL'}")

In [ ]:
# ── Paso 2: SHAP ───────────────────────────────────────────────────
explainer2 = shap.TreeExplainer(model2)
shap_raw2 = explainer2.shap_values(X_te2)

# Normalizar formato
if isinstance(shap_raw2, list):
    shap_vals2 = shap_raw2
else:
    arr = np.array(shap_raw2)
    if arr.ndim == 3 and arr.shape[2] == len(RETAILERS_TARGET):
        shap_vals2 = [arr[:, :, c] for c in range(arr.shape[2])]
    elif arr.ndim == 3 and arr.shape[0] == len(RETAILERS_TARGET):
        shap_vals2 = [arr[c] for c in range(arr.shape[0])]
    else:
        raise ValueError(f"SHAP shape inesperado: {arr.shape}")

mean_shap2 = np.mean([np.abs(sv) for sv in shap_vals2], axis=0).mean(axis=0)
imp2 = pd.Series(mean_shap2, index=FEATURE_COLS).nlargest(10)

fig, ax = plt.subplots(figsize=(8, 5))
imp2.sort_values().plot(kind='barh', color='steelblue', ax=ax)
ax.set_title('Paso 2: Top 10 Features (SHAP)', fontsize=13)
plt.tight_layout()
plt.show()

print("Top 10 features Paso 2:")
for f, v in imp2.items():
    print(f"  {f}: {v:.4f}")

---
## 3. Paso 3: ¿Cuánto canjea? (Monto en puntos)

XGBoost regresión sobre **solo canjeadores**.  
Target: `monto_redeem_post` (puntos totales canjeados en 12m post-t0).  
Predicción condicional: solo se ejecuta si el cliente supera el threshold de Paso 1.

| Métrica | Mínimo | Objetivo |
|---------|--------|----------|
| R² | > 0.40 | > 0.60 |
| MAPE | < 35% | < 20% |

In [ ]:
# ── Paso 3: Preparar datos (canjeadores con monto > 0) ───────────
tr3 = df_train[df_train['monto_redeem_post'].notna()].copy()
va3 = df_val[df_val['monto_redeem_post'].notna()].copy()
te3 = df_test[df_test['monto_redeem_post'].notna()].copy()

X_tr3 = tr3[FEATURE_COLS].values.astype(np.float32)
y_tr3 = tr3['monto_redeem_post'].values.astype(np.float32)
X_va3 = va3[FEATURE_COLS].values.astype(np.float32)
y_va3 = va3['monto_redeem_post'].values.astype(np.float32)
X_te3 = te3[FEATURE_COLS].values.astype(np.float32)
y_te3 = te3['monto_redeem_post'].values.astype(np.float32)

print(f"Paso 3 — Train: {len(tr3)}, Val: {len(va3)}, Test: {len(te3)}")
print(f"Monto train: mean={y_tr3.mean():,.0f}, median={np.median(y_tr3):,.0f}, std={y_tr3.std():,.0f}")

In [ ]:
# ── Paso 3: Optuna (optimizar RMSE en val) ─────────────────────
def objective_paso3(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 800),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10, log=True),
        'tree_method': 'hist',
        'objective': 'reg:squarederror',
        'random_state': 42,
        'verbosity': 0,
        'early_stopping_rounds': 50,
    }
    model = xgb.XGBRegressor(**params)
    model.fit(X_tr3, y_tr3, eval_set=[(X_va3, y_va3)], verbose=False)
    y_pred = model.predict(X_va3)
    return -np.sqrt(mean_squared_error(y_va3, y_pred))  # minimize RMSE

study3 = optuna.create_study(direction='maximize', study_name='paso3_monto')
study3.optimize(objective_paso3, n_trials=30, show_progress_bar=True)

print(f"\nBest RMSE (val): {-study3.best_value:,.0f}")
print(f"Best params: {study3.best_params}")

In [ ]:
# ── Paso 3: Modelo final + evaluacion ────────────────────────────
best_p3 = study3.best_params.copy()
best_p3.update({'tree_method': 'hist', 'objective': 'reg:squarederror',
                'random_state': 42, 'verbosity': 0, 'early_stopping_rounds': 50})

model3 = xgb.XGBRegressor(**best_p3)
model3.fit(X_tr3, y_tr3, eval_set=[(X_va3, y_va3)], verbose=False)

y_pred3 = model3.predict(X_te3)
y_pred3 = np.maximum(y_pred3, 0)  # monto no puede ser negativo

r2_3 = r2_score(y_te3, y_pred3)
mask_nonzero = y_te3 > 0
mape_3 = np.mean(np.abs(y_te3[mask_nonzero] - y_pred3[mask_nonzero]) / y_te3[mask_nonzero]) * 100
rmse_3 = np.sqrt(mean_squared_error(y_te3, y_pred3))

print(f"TEST — R\u00b2: {r2_3:.4f}, MAPE: {mape_3:.1f}%, RMSE: {rmse_3:,.0f}")

# Scatter plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(y_te3, y_pred3, alpha=0.4, s=20)
lim = max(y_te3.max(), y_pred3.max()) * 1.1
axes[0].plot([0, lim], [0, lim], 'r--', alpha=0.5)
axes[0].set_xlabel('Actual (puntos)')
axes[0].set_ylabel('Predicho (puntos)')
axes[0].set_title(f'Paso 3: Actual vs Predicho (R\u00b2={r2_3:.3f})')

residuos = y_te3 - y_pred3
axes[1].hist(residuos, bins=30, color='coral', edgecolor='white')
axes[1].axvline(0, color='black', ls='--')
axes[1].set_title('Residuos (puntos)')
plt.tight_layout()
plt.show()

print(f"\n{'Metrica':<25} {'Minimo':<10} {'Objetivo':<10} {'Resultado':<10} {'Status'}")
print('=' * 65)
print(f"{'R2':<25} {'>0.40':<10} {'>0.60':<10} {r2_3:<10.4f} {'OK' if r2_3 > 0.40 else 'FAIL'}")
print(f"{'MAPE':<25} {'<35%':<10} {'<20%':<10} {f'{mape_3:.1f}%':<10} {'OK' if mape_3 < 35 else 'FAIL'}")

In [ ]:
# ── Paso 3: SHAP ───────────────────────────────────────────────────
explainer3 = shap.TreeExplainer(model3)
shap_vals3 = explainer3.shap_values(X_te3)

imp3 = pd.Series(np.abs(shap_vals3).mean(axis=0), index=FEATURE_COLS).nlargest(10)

fig, ax = plt.subplots(figsize=(8, 5))
imp3.sort_values().plot(kind='barh', color='coral', ax=ax)
ax.set_title('Paso 3: Top 10 Features (SHAP)', fontsize=13)
plt.tight_layout()
plt.show()

print("Top 10 features Paso 3:")
for f, v in imp3.items():
    print(f"  {f}: {v:.4f}")

---
## 4. Paso 4: ¿Cuánto revenue genera? ($CLP)

XGBoost regresión sobre **TODOS los clientes**.  
Target: `revenue_post_12m` (gasto total $CLP en 12m post-t0).  
Incluye clientes con revenue=0.

| Métrica | Mínimo | Objetivo |
|---------|--------|----------|
| R² | > 0.40 | > 0.60 |
| MAPE | < 30% | < 18% |

In [ ]:
# ── Paso 4: Preparar datos (TODOS los clientes) ─────────────────
X_tr4 = df_train[FEATURE_COLS].values.astype(np.float32)
y_tr4 = df_train['revenue_post_12m'].values.astype(np.float32)
X_va4 = df_val[FEATURE_COLS].values.astype(np.float32)
y_va4 = df_val['revenue_post_12m'].values.astype(np.float32)
X_te4 = df_test[FEATURE_COLS].values.astype(np.float32)
y_te4 = df_test['revenue_post_12m'].values.astype(np.float32)

print(f"Paso 4 — Train: {len(X_tr4)}, Val: {len(X_va4)}, Test: {len(X_te4)}")
print(f"Revenue train: mean={y_tr4.mean():,.0f}, median={np.median(y_tr4):,.0f}, zeros={(y_tr4==0).sum()} ({(y_tr4==0).mean()*100:.1f}%)")

In [ ]:
# ── Paso 4: Optuna (optimizar RMSE en val) ─────────────────────
def objective_paso4(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 800),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10, log=True),
        'tree_method': 'hist',
        'objective': 'reg:squarederror',
        'random_state': 42,
        'verbosity': 0,
        'early_stopping_rounds': 50,
    }
    model = xgb.XGBRegressor(**params)
    model.fit(X_tr4, y_tr4, eval_set=[(X_va4, y_va4)], verbose=False)
    y_pred = model.predict(X_va4)
    return -np.sqrt(mean_squared_error(y_va4, y_pred))

study4 = optuna.create_study(direction='maximize', study_name='paso4_revenue')
study4.optimize(objective_paso4, n_trials=30, show_progress_bar=True)

print(f"\nBest RMSE (val): {-study4.best_value:,.0f}")
print(f"Best params: {study4.best_params}")

In [ ]:
# ── Paso 4: Modelo final + evaluacion ────────────────────────────
best_p4 = study4.best_params.copy()
best_p4.update({'tree_method': 'hist', 'objective': 'reg:squarederror',
                'random_state': 42, 'verbosity': 0, 'early_stopping_rounds': 50})

model4 = xgb.XGBRegressor(**best_p4)
model4.fit(X_tr4, y_tr4, eval_set=[(X_va4, y_va4)], verbose=False)

y_pred4 = model4.predict(X_te4)
y_pred4 = np.maximum(y_pred4, 0)  # revenue no negativo

r2_4 = r2_score(y_te4, y_pred4)
# MAPE excluyendo revenue=0
mask_nz = y_te4 > 0
mape_4 = np.mean(np.abs(y_te4[mask_nz] - y_pred4[mask_nz]) / y_te4[mask_nz]) * 100 if mask_nz.any() else float('nan')
rmse_4 = np.sqrt(mean_squared_error(y_te4, y_pred4))

print(f"TEST — R\u00b2: {r2_4:.4f}, MAPE: {mape_4:.1f}% (excl. zeros), RMSE: {rmse_4:,.0f}")

# Scatter + residuos
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(y_te4, y_pred4, alpha=0.3, s=15)
lim4 = max(y_te4.max(), y_pred4.max()) * 1.1
axes[0].plot([0, lim4], [0, lim4], 'r--', alpha=0.5)
axes[0].set_xlabel('Actual ($CLP)')
axes[0].set_ylabel('Predicho ($CLP)')
axes[0].set_title(f'Paso 4: Actual vs Predicho (R\u00b2={r2_4:.3f})')

residuos4 = y_te4 - y_pred4
axes[1].hist(residuos4, bins=30, color='seagreen', edgecolor='white')
axes[1].axvline(0, color='black', ls='--')
axes[1].set_title('Residuos ($CLP)')
plt.tight_layout()
plt.show()

print(f"\n{'Metrica':<25} {'Minimo':<10} {'Objetivo':<10} {'Resultado':<10} {'Status'}")
print('=' * 65)
print(f"{'R2':<25} {'>0.40':<10} {'>0.60':<10} {r2_4:<10.4f} {'OK' if r2_4 > 0.40 else 'FAIL'}")
print(f"{'MAPE':<25} {'<30%':<10} {'<18%':<10} {f'{mape_4:.1f}%':<10} {'OK' if mape_4 < 30 else 'FAIL'}")

In [ ]:
# ── Paso 4: SHAP ───────────────────────────────────────────────────
explainer4 = shap.TreeExplainer(model4)
shap_vals4 = explainer4.shap_values(X_te4)

imp4 = pd.Series(np.abs(shap_vals4).mean(axis=0), index=FEATURE_COLS).nlargest(10)

fig, ax = plt.subplots(figsize=(8, 5))
imp4.sort_values().plot(kind='barh', color='seagreen', ax=ax)
ax.set_title('Paso 4: Top 10 Features (SHAP)', fontsize=13)
plt.tight_layout()
plt.show()

print("Top 10 features Paso 4:")
for f, v in imp4.items():
    print(f"  {f}: {v:.4f}")

---
## 5. Cascade Integrado

Tabla resumen de los 4 modelos + pipeline end-to-end.

In [ ]:
# ── Tabla resumen 4 modelos ──────────────────────────────────────
print("="*80)
print("RESUMEN CASCADE PREDICTIVO — 4 MODELOS")
print("="*80)
print(f"\n{'Paso':<8} {'Tipo':<15} {'Target':<20} {'Metrica clave':<20} {'Resultado':<12} {'Status'}")
print("-"*80)
print(f"{'Paso 1':<8} {'Multiclase':<15} {'y (0/1/2)':<20} {'F1-macro':<20} {'[fase2a]':<12} {'--'}")
print(f"{'Paso 2':<8} {'Multiclase':<15} {'retailer_post':<20} {'Accuracy':<20} {acc2:<12.4f} {'OK' if acc2 > 0.60 else 'FAIL'}")
print(f"{'     ':<8} {'          ':<15} {'             ':<20} {'Top-2 Acc':<20} {top2_acc:<12.4f} {'OK' if top2_acc > 0.85 else 'FAIL'}")
print(f"{'Paso 3':<8} {'Regresion':<15} {'monto (pts)':<20} {'R2':<20} {r2_3:<12.4f} {'OK' if r2_3 > 0.40 else 'FAIL'}")
print(f"{'     ':<8} {'         ':<15} {'           ':<20} {'MAPE':<20} {f'{mape_3:.1f}%':<12} {'OK' if mape_3 < 35 else 'FAIL'}")
print(f"{'Paso 4':<8} {'Regresion':<15} {'revenue (CLP)':<20} {'R2':<20} {r2_4:<12.4f} {'OK' if r2_4 > 0.40 else 'FAIL'}")
print(f"{'     ':<8} {'         ':<15} {'             ':<20} {'MAPE':<20} {f'{mape_4:.1f}%':<12} {'OK' if mape_4 < 30 else 'FAIL'}")
print()
print("Nota: Con datos mock (1000 clientes), las metricas son orientativas.")
print("En produccion (500K clientes x 27 t0s) se esperan resultados significativamente mejores.")

In [ ]:
# ── Pipeline cascade end-to-end (ejemplo con 5 clientes del test) ───
# Simular: cargar modelo Paso 1 de fase2a (aqui reconstruimos rapido)
# Para demo, usamos las probabilidades reales de y como proxy

THRESHOLD_CANJE = 0.3  # P(y=1) + P(y=2) > 0.3

# Muestra de 10 clientes del test
sample_idx = np.random.RandomState(42).choice(len(df_test), 10, replace=False)
sample = df_test.iloc[sample_idx].copy()
X_sample = sample[FEATURE_COLS].values.astype(np.float32)

# Paso 1: P(canje) — usar y real como proxy (en prod viene del modelo fase2a)
p_canje_proxy = (sample['y'].isin([1, 2])).astype(float).values
# En produccion: p_canje = model1.predict_proba(X_sample)[:, 1] + model1.predict_proba(X_sample)[:, 2]

# Paso 2: retailer (solo si p_canje > threshold)
mask_canje = p_canje_proxy > THRESHOLD_CANJE
retailer_pred = np.full(len(sample), '-', dtype=object)
if mask_canje.any():
    proba_ret = model2.predict_proba(X_sample[mask_canje])
    retailer_pred[mask_canje] = le_ret.inverse_transform(proba_ret.argmax(axis=1))

# Paso 3: monto (solo si p_canje > threshold)
monto_pred = np.full(len(sample), 0.0)
if mask_canje.any():
    monto_pred[mask_canje] = np.maximum(model3.predict(X_sample[mask_canje]), 0)

# Paso 4: revenue (todos)
revenue_pred = np.maximum(model4.predict(X_sample), 0)

# Output
output = pd.DataFrame({
    'cust_id': sample['cust_id'].values,
    'y_real': sample['y'].values,
    'P(canje)': p_canje_proxy,
    'Pasa_threshold': mask_canje,
    'retailer_pred': retailer_pred,
    'retailer_real': sample['retailer_post'].fillna('-').values,
    'monto_pred': monto_pred.astype(int),
    'monto_real': sample['monto_redeem_post'].fillna(0).astype(int).values,
    'revenue_pred': revenue_pred.astype(int),
    'revenue_real': sample['revenue_post_12m'].astype(int).values,
})

print("Pipeline Cascade — Ejemplo con 10 clientes del TEST:")
print(output.to_string(index=False))

---
## Resumen y Next Steps

### Resultados Cascade Completo (Mock)

| Paso | Modelo | Target | Métrica clave | Status |
|------|--------|--------|---------------|--------|
| 1 | XGB multiclase | y (0/1/2) | F1-macro=0.709 | fase2a |
| 2 | XGB multiclase | retailer | Accuracy / Top-2 | este notebook |
| 3 | XGB regresión | monto (pts) | R² / MAPE | este notebook |
| 4 | XGB regresión | revenue (CLP) | R² / MAPE | este notebook |

### Next Steps
- **Fase 2c**: Clustering / Segmentación (KMeans/GMM)
- **Fase 2d**: Incrementalidad (PSM + Uplift)
- **Fase 2e**: Decision Engine
- **Producción**: Cambiar `USE_MOCK=False` y conectar a BigQuery